# Get to Know a Dataset: AOT (Arrow of Time) 7T fMRI

This notebook is a guided tour of the [Arrow of Time (AOT)](https://registry.opendata.aws/REGISTRY_YAML_NAME)
dataset: a 7-tesla fMRI study in which observers watched short natural movie
clips played **forwards** and **backwards**, designed to ask how visual cortex
represents the temporal direction of natural events.

It comes in two halves:

1. **Getting the data** — how the dataset is laid out in S3, and how to pull
   down the slice you actually need.
2. **Using the data** — how to read what you downloaded with
   [`AOTaccess`](https://github.com/AOT-Spinoza/aot_access), the dataset's
   Python API, which turns BIDS-style paths into arrays so you never hand-build
   a filename.

More usage examples, tutorials, and documentation for this dataset and others
can be found at the [Registry of Open Data on AWS](https://registry.opendata.aws/).

## What is in the dataset

Eight subjects each viewed 2,160 unique one-second movie clips, every clip
shown once forwards and once backwards, across twelve 2-hour scanning
sessions. Alongside the main experiment each subject has a population
receptive field (pRF) mapping session and a category localizer, plus
anatomical scans.

| | |
|---|---|
| Subjects | 8 (`sub-001` … `sub-008`) |
| Main-task sessions | 10–12 per subject, 10 runs each |
| Unique video clips | 2,179 (each forwards + backwards) |
| Trials per subject | ~10,000 |
| Field strength | 7 T |
| EPI resolutions | 2.0 mm and 1.25 mm isotropic |

The published dataset contains the raw BIDS data *and* the derivatives most
users actually want — preprocessed timeseries, single-trial GLM estimates,
pRF fits, ROI definitions, and stimulus annotations — so that you can start
from analysis-ready arrays rather than rerunning a preprocessing pipeline.

## Q: How have you organized your dataset? Help us understand the key prefix structure of your S3 bucket.

The bucket mirrors the on-disk layout of the dataset exactly, so anything you
download keeps working with the dataset's own tooling. There are two
top-level prefixes:

- **`bids/aot/`** — the raw dataset in [BIDS](https://bids.neuroimaging.io/)
  format: one prefix per subject, then per session, then the usual
  `anat/`, `func/`, `fmap/` folders.
- **`derivatives/`** — everything computed from the raw data:

| Prefix | What it holds |
|---|---|
| `derivatives/aot_prep/` | Preprocessed BOLD timeseries |
| `derivatives/glmsingle/` | Single-trial beta estimates (per session and per video) |
| `derivatives/localizers/prf/` | Population receptive field model fits |
| `derivatives/roi/` | ROI masks and parcellations, per subject and group |
| `derivatives/anat-3T/` | Anatomies resampled onto each EPI grid |
| `derivatives/stimuli/` | The movie clips, plus frames and annotations |

Two conventions are worth knowing before you download anything.

**Spaces and resolutions.** Volumetric derivatives carry
`space-T1w` (the subject's own anatomy, which is also the grid the functional
data live on) or `space-MNI`, together with `res-2p0mm` or `res-1p25mm`. The
practical consequence: an ROI mask in `space-T1w` at `res-2p0mm` indexes that
subject's `res-2p0mm` betas directly, with no resampling step.

**Per-session vs per-video betas.** `glmsingle/per_session/` stores one file
per scanning session with every trial in acquisition order. `glmsingle/per_video/`
reorganises the same estimates by stimulus, so all responses to clip `0042`
forwards sit in one file. Choose whichever matches your analysis.

In [ ]:
# This notebook requires the following additional libraries
# (please install using the preferred method for your environment, e.g. pip, conda):
#
# boto3 >= 1.38.23
# aot-access >= 0.1.0
# nibabel >= 5.3.0
# numpy >= 2.1.0
# matplotlib >= 3.10.3

# Built-ins
from pathlib import Path

# Installed libraries
import boto3
import numpy as np
import matplotlib.pyplot as plt
from botocore import UNSIGNED
from botocore.config import Config as BotoConfig

In [ ]:
# The S3 bucket holding the dataset. AOT is a public dataset, so requests are
# unsigned — no AWS credentials are needed to read it.
BUCKET = "REPLACE-WITH-AOT-BUCKET-NAME"

# Where the downloaded slice of the dataset will live. Keeping the bucket's
# internal layout intact under this directory is what lets AOTaccess read it
# later without any further configuration.
LOCAL_ROOT = Path("./aot-data").resolve()

s3 = boto3.client("s3", config=BotoConfig(signature_version=UNSIGNED))

Listing the top of the bucket shows the two top-level prefixes described above.

In [ ]:
def list_prefixes(prefix=""):
    """List the immediate sub-prefixes under an S3 prefix."""
    paginator = s3.get_paginator("list_objects_v2")
    pages = paginator.paginate(Bucket=BUCKET, Prefix=prefix, Delimiter="/")
    return [cp["Prefix"] for page in pages
            for cp in page.get("CommonPrefixes", [])]


for prefix in list_prefixes():
    print(prefix)

And one level deeper, the derivative stores:

In [ ]:
for prefix in list_prefixes("derivatives/"):
    print(prefix)

## Q: What data formats are present in your dataset?

Five formats cover the whole dataset, all of them standard in neuroimaging or
general data science:

**NIfTI (`.nii.gz`)** is the standard container for volumetric brain imaging:
a 3-D or 4-D array plus an *affine* matrix mapping voxel indices to physical
coordinates in millimetres. Every image here is one — anatomies and masks are
3-D, timeseries and multi-trial beta files are 4-D. Read them with
[NiBabel](https://nipy.org/nibabel/); `img.get_fdata()` gives you a NumPy
array and `img.affine` the spatial mapping. Because all of a subject's
volumetric derivatives share one grid, you can index one array with a mask
from another without resampling.

**TSV (`.tsv`)** carries tabular data — BIDS event files describing what was
shown when, confound regressors, and pRF parameter tables. Plain text, one
header row; `pandas.read_csv(path, sep="\t")` reads them.

**YAML (`.yaml`)** holds machine-readable metadata. Notably, the ROI and
localizer stores each ship a manifest describing their own contents and
filename templates, which is how the API resolves paths without hard-coding
them.

**HDF5 (`.h5`)** stores the motion-energy features extracted from each clip —
arrays too large and too numerous for text. Read with
[h5py](https://www.h5py.org/).

**MP4 (`.mp4`)** is the stimuli themselves, one file per clip per direction.

We chose these because every one is an established standard with mature
readers in Python, R and MATLAB, and because BIDS-compliant NIfTI and TSV mean
the raw data works directly with the wider neuroimaging ecosystem
(fMRIPrep, FSL, AFNI, nilearn) rather than requiring a bespoke loader.

At AWS scale, [Amazon Athena](https://docs.aws.amazon.com/athena/latest/ug/)
can query the TSV tables in place, and
[AWS Batch](https://docs.aws.amazon.com/batch/) or
[SageMaker](https://docs.aws.amazon.com/sagemaker/) suit the
embarrassingly-parallel per-subject workloads this dataset invites.

## Q: Can you show us an example of downloading and loading data from your dataset?

The complete dataset is large, so the useful pattern is a **targeted
download**: pick a subject and the specific derivatives your analysis needs,
and pull only those prefixes.

For this tour we take one subject (`sub-006`) and three things: the
single-trial betas for one session, that subject's ROI masks, and the anatomy
that shares their functional grid.

In [ ]:
def download_prefix(prefix, dest_root=LOCAL_ROOT, dry_run=False):
    """Download every object under an S3 prefix, preserving its layout.

    Returns (n_objects, total_bytes). With dry_run=True nothing is written,
    which is a cheap way to check the size of a prefix before committing to it.
    """
    paginator = s3.get_paginator("list_objects_v2")
    n, total = 0, 0
    for page in paginator.paginate(Bucket=BUCKET, Prefix=prefix):
        for obj in page.get("Contents", []):
            key = obj["Key"]
            if key.endswith("/"):
                continue
            n, total = n + 1, total + obj["Size"]
            if dry_run:
                continue
            target = dest_root / key
            target.parent.mkdir(parents=True, exist_ok=True)
            if not target.exists():            # resumable: skip what we have
                s3.download_file(BUCKET, key, str(target))
    return n, total

Which slice of the dataset this tour uses. Change these to retarget everything
below — the rest of the notebook is written in terms of them.

In [ ]:
SUB = "sub-006"      # any of sub-001 ... sub-008
SES = "ses-01"       # main-task sessions run ses-01 ... ses-12
RES = "2p0mm"        # "2p0mm" or "1p25mm"

Before downloading, check how big each prefix is. `dry_run=True` lists without
fetching.

In [ ]:
wanted = {
    "session betas": f"derivatives/glmsingle/per_session/{SUB}/{SES}/space-T1w_res-{RES}/",
    "ROI masks":     f"derivatives/roi/{SUB}/fsnative_volume/atlas-wang_2015/res-{RES}/",
    "anatomy":       f"derivatives/anat-3T/{SUB}/fiducial/res-{RES}/",
}

for label, prefix in wanted.items():
    n, total = download_prefix(prefix, dry_run=True)
    print(f"{label:>16}: {n:5d} objects, {total / 1e6:8.1f} MB")

The ROI store is manifest-driven, so we also need its `MANIFEST.yaml` — that
file is what tells the API where every mask lives.

In [ ]:
# Fetch the small manifests that describe the stores we downloaded
for key in ["derivatives/roi/MANIFEST.yaml"]:
    target = LOCAL_ROOT / key
    target.parent.mkdir(parents=True, exist_ok=True)
    s3.download_file(BUCKET, key, str(target))

# ... then the data itself
for label, prefix in wanted.items():
    n, total = download_prefix(prefix)
    print(f"downloaded {label}: {n} objects ({total / 1e6:.1f} MB)")

## Using what you downloaded, with AOTaccess

Everything above is generic S3. From here on we use
[`AOTaccess`](https://github.com/AOT-Spinoza/aot_access), the dataset's own
read-only Python API (`pip install aot-access`).

The reason to bother: filenames in this dataset encode subject, session,
space, resolution, model and descriptor, and getting one wrong gives you a
confusing `FileNotFoundError` rather than a useful message. `AOTaccess` builds
them for you, and returns arrays already reshaped for analysis.

The bridge from the download to the API is one argument. Point `root_path` at
the directory you downloaded into and every store resolves beneath it — no
configuration file, no environment variable, and nothing reaching back to the
original cluster. This works precisely because the download preserved the
bucket's layout.

In [ ]:
from AOTaccess.aot_access import AOTAccess

# Point the API at the downloaded copy rather than a cluster installation
aot = AOTAccess(root_path=LOCAL_ROOT)

# Everything resolves relative to that root
print(aot.config.path("glmsingle"))
print(aot.config.path("roi"))
print(aot.config.anatomy_root())

### The subject object

`AOTSubject` is the main entry point. Bind a subject once and ask for data in
analysis-ready shape.

Its organising idea is a **brain mask**: rather than hand you 3-D volumes full
of empty space, it flattens every array over a per-subject mask of interesting
voxels. The default is the dilated FreeSurfer cortical grey matter, which
keeps low-signal regions like the default mode network in play instead of
silently discarding them.

In [ ]:
subject = aot.subject(SUB)

mask = subject.get_brain_mask()
print(f"brain mask: {mask.shape} volume, {mask.sum():,} voxels in the working set")

### Single-trial responses

`get_betas` returns `(n_trials, n_voxels)` — trials first, the convention
every scikit-learn-style estimator expects. Restrict it to an ROI by name and
the voxel axis shrinks to that region.

In [ ]:
betas_all = subject.get_betas(ses=1)
betas_v1 = subject.get_betas(ses=1, roi="V1v")

print(f"whole working set : {betas_all.shape}   (trials x voxels)")
print(f"V1v only          : {betas_v1.shape}")

### What was shown on each trial

The betas are just numbers until you know what each trial contained.
`trial_table` returns one row per trial, with the clip identifier, the
direction it was played in, and whether it was a blank trial — the design
matrix for whatever model you fit next.

In [ ]:
trials = subject.trial_table()
print(f"{len(trials):,} trials for {SUB}")
trials.head()

Those two pieces are all you need to pose the dataset's central question.
Every clip appears once forwards and once backwards, so the `direction`
column splits the trials into the contrast the study was built around:

In [ ]:
session_one = trials[trials.ses == 1]
stim = session_one[~session_one.is_blank]

print(f"session 1: {len(session_one)} trials "
      f"({len(stim)} clips, {len(session_one) - len(stim)} blanks)")
print(stim.direction.value_counts().to_string())

### Regions of interest

ROI masks come from a manifest-driven library covering several published
atlases. They are defined on the same grid as the functional data, so a mask
indexes the betas directly.

In [ ]:
roi = aot.roi_access

print("atlases available:", ", ".join(roi.atlases()))
print()

v1v = subject.get_roi_mask("V1v")
print(f"V1v: {v1v.sum():,} voxels within the brain mask")

## Q: A picture is worth a thousand words.

A quick look at the data we just loaded. On the left, where V1v sits in the
brain; on the right, the distribution of single-trial responses in that
region — centred near zero, as single-trial GLM estimates should be, with the
spread that makes trial-by-trial decoding possible.

In [ ]:
fig, (ax_brain, ax_hist) = plt.subplots(1, 2, figsize=(13, 5))

# Where V1v is: a mid-sagittal-ish slice with the ROI overlaid on the mask
z = int(np.argmax(v1v.sum(axis=(0, 1))))          # slice with most V1v voxels
ax_brain.imshow(mask[:, :, z].T, cmap="Greys", origin="lower")
ax_brain.imshow(np.ma.masked_where(~v1v[:, :, z], v1v[:, :, z]).T,
                cmap="autumn", origin="lower", interpolation="nearest")
ax_brain.set_title(f"V1v within the working set (slice z={z})",
                   fontsize=13, fontweight="bold")
ax_brain.axis("off")

# How single-trial responses are distributed in that ROI
ax_hist.hist(betas_v1.ravel(), bins=60, color="#3498db",
             edgecolor="white", linewidth=0.8, alpha=0.85)
ax_hist.set_title("Single-trial responses in V1v", fontsize=13,
                  fontweight="bold")
ax_hist.set_xlabel("beta (% signal change)", fontsize=11)
ax_hist.set_ylabel("count", fontsize=11)
ax_hist.grid(True, linestyle="--", alpha=0.3)
ax_hist.set_facecolor("#f8f9fa")
for side in ("top", "right"):
    ax_hist.spines[side].set_visible(False)

plt.tight_layout()
plt.show()

## Q: What is one question that you have answered using these data?

**Does visual cortex distinguish a clip played forwards from the same clip
played backwards?**

This is the question the dataset was built for, and its design makes the test
unusually clean. Because every clip is shown in both directions, the forwards
and backwards conditions contain *identical* low-level content — the same
frames, the same colours, the same spatial statistics, merely reordered. Any
reliable difference in response cannot be explained by those properties.

Below is the smallest honest version of the analysis: within one session and
one region, is the mean V1v response different for the two directions? We
average over voxels to get one number per trial and compare the groups.

In [ ]:
# GLMsingle estimates one beta per *stimulus* trial — blank trials are not
# modelled — so the betas line up with the non-blank rows of the trial table,
# in the same chronological order.
assert len(stim) == len(betas_v1)

response = betas_v1.mean(axis=1)          # one number per trial
direction = stim.direction.to_numpy()

fw = response[direction == "fw"]
rv = response[direction == "rv"]

print(f"forwards : n={len(fw):4d}   mean={fw.mean():+.4f}   sd={fw.std():.4f}")
print(f"backwards: n={len(rv):4d}   mean={rv.mean():+.4f}   sd={rv.std():.4f}")
print(f"difference (fw - rv): {fw.mean() - rv.mean():+.4f}")

A univariate mean like this is a deliberately blunt instrument, and in early
visual cortex it is the *expected* place to find little or nothing: V1 responds
to the frames, and both conditions contain the same frames. The interesting
signal in this dataset lives in two directions the snippet above does not
go — in **multivariate** patterns across voxels rather than their average, and
in **higher** areas where temporal structure is represented. Scaling the same
two lines across all twelve sessions, all subjects, and the full ROI set is
what turns this into a real result, and is exactly what `get_betas` and
`get_roi_mask` are designed to make routine.

## Q: What is one unanswered question that you think could be answered using these data?

**Where does the brain stop tracking what is happening and start tracking
whether it could happen in that order?**

A clip played backwards is not unnatural everywhere in the same way. Smoke
drifting in reverse is merely unusual; a shattered glass reassembling is
physically impossible. If some regions are sensitive to violations of
*physical plausibility* rather than to temporal reordering as such, then those
regions should distinguish forwards from backwards much more strongly for
clips depicting irreversible events than for clips whose reversal remains
plausible.

Testing this needs the clips grouped by how reversible the event they depict
actually is — and the dataset ships the raw material for that grouping, in the
form of per-clip semantic annotations and human behavioural judgements
alongside the imaging data.

Some suggestions for anyone taking this on:

- **Start with the behaviour.** The dataset includes online judgements of how
  identifiable each clip's direction is. Those ratings give you an empirical
  reversibility axis, rather than one you had to define by hand.
- **Control the low-level story.** Motion energy features ship with the
  dataset for exactly this purpose: show that whatever you find survives
  matching clips on motion statistics.
- **Let the ROI hierarchy do the work.** The atlases included here span early
  visual cortex through to association areas, so "where does this transition
  happen" is a question you can ask directly.
- **Mind the multiple comparisons.** Many ROIs times many clip groupings adds
  up quickly; hold out subjects, or pre-register the contrast.

The pRF fits included for every subject let you ask the same question while
controlling for where each voxel is looking — a level of rigour that is
usually out of reach without a dedicated mapping session.

## Where to go next

- **The API documentation** covers the readers this notebook skipped: pRF
  parameter maps, preprocessed timeseries, stimulus annotations, and
  cross-subject access via `AOTGroup`.
- **`discovery`** answers "what is actually present?" —
  `discovery.subjects()`, `discovery.sessions(6)`, `discovery.videos(6)` —
  which is the fastest way to find out what a given download contains.
- **`per_video` betas** reorganise the same estimates by stimulus rather than
  by session, which is usually what you want for encoding models.

If you use this dataset, please cite it — see the Registry of Open Data
landing page for the current reference.